# Rice Disease Detection — AI AgriData Intelligence Race (TELEPATI 8.0)

Notebook ini memuat seluruh alur pengembangan model deteksi penyakit padi, mulai dari dataset mentah resmi sampai inference dan perhitungan skor.

| | |
|---|---|
| Tugas | Object detection 11 kelas (10 penyakit + Healthy), citra 640×640 |
| Data | Dataset resmi Agritech Data (format COCO, split train/valid/test) |
| Model | Ultralytics YOLO, dibangun dari file `.yaml` dan dilatih dari nol |
| Metrik | F1-Score makro pada split test resmi |
| Platform | Google Colab, GPU T4 |

**Mode eksekusi**

| `MODE` | Yang dijalankan | Kegunaan |
|---|---|---|
| `"full"` | Bagian 1–10 | Reproduksi lengkap: data mentah → training → evaluasi |
| `"inference_only"` | Bagian 1–3, lalu 11 | Reproduksi skor dari weights rilis, tanpa training |

**Kepatuhan aturan**

- Seluruh pemrosesan data dimulai dari zip dataset mentah dan dikerjakan di notebook ini.
- Training hanya memakai split **train**. Split **valid** dipakai untuk memilih checkpoint. Split **test** hanya dipakai di bagian 10 (sekali) dan bagian 11.
- Arsitektur dibangun dari file `.yaml` dengan `pretrained=False`. Tidak ada bobot eksternal yang dipakai untuk training.
- Seed tetap (`SEED = 42`) dan mode deterministik di setiap tahap.

## 0. Konfigurasi

Semua parameter yang perlu diubah ada di sel ini. Sel lain tidak perlu disunting.

- `RUN_ID` memilih konfigurasi training dari tabel `RUNS` di bagian 7.
- `WEIGHTS_URL`, `WEIGHTS_SHA256`, dan `E2_CONF` diisi setelah model final dirilis.
- `RUN_FINAL_TEST` dibiarkan `False` sampai model final dipilih dari hasil valid.

In [ ]:
MODE = "full"  # "full" | "inference_only"
RUN_ID = "A_yolo11s_s42"

DATA_ZIP = "/content/drive/MyDrive/telepati/data/telepati-agridata-raw.zip"
DATA_ZIP_SHA256 = "797e47749f91f4403e778dd0150716da1cd94cddbecee2ec56cf5545c50cec50"  # zip referensi tim

RUNS_DIR = "/content/drive/MyDrive/telepati/runs"
WORK_DIR = "/content/work"

WEIGHTS_URL = ""
WEIGHTS_SHA256 = ""
E2_CONF = None

SEED = 42
IMGSZ = 640
EVAL_BATCH = 16
RUN_FINAL_TEST = False

## 1. Lingkungan

Memasang versi library yang dipin, menghubungkan Google Drive (khusus Colab), lalu mengunci seed.

Catatan: saat training dengan AMP, Ultralytics mengunduh model kecil hanya untuk *memeriksa* dukungan mixed precision pada GPU. Bobot tersebut tidak dipakai sebagai inisialisasi; model tetap dibangun dari `.yaml`.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.184"], check=True)
    from google.colab import drive

    drive.mount("/content/drive")

In [ ]:
import hashlib
import json
import math
import os
import random
import shutil
import time
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml
from ultralytics import YOLO

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

WORK_DIR = Path(WORK_DIR)
RUNS_DIR = Path(RUNS_DIR)
EXTRACT_DIR = WORK_DIR / "raw"
YOLO_DIR = WORK_DIR / "yolo"
SPLITS = ("train", "valid", "test")

print(f"python      {sys.version.split()[0]}")
print(f"torch       {torch.__version__}")
print(f"ultralytics {ultralytics.__version__}")
print(f"device      {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 2. Dataset mentah

Notebook menerima dua bentuk zip:

- zip resmi dari panitia, yang membungkus split dalam satu folder dan menyertakan file metadata macOS (`__MACOSX`, `._*`, `.DS_Store`);
- zip kerja tim, yang langsung berisi `train/`, `valid/`, dan `test/`.

File metadata macOS dilewati saat ekstraksi, lalu folder yang berisi ketiga split dicari otomatis. Ekstraksi dianggap selesai hanya jika setiap gambar yang tercantum di file COCO benar-benar ada, sehingga ekstraksi yang sempat terputus diulang dari awal. Checksum dibandingkan dengan zip referensi tim sebagai informasi; validitas isi dipastikan lewat jumlah gambar dan anotasi per split.

In [ ]:
EXPECTED_COUNTS = {
    "train": {"images": 10133, "annotations": 20163},
    "valid": {"images": 2106, "annotations": 4888},
    "test": {"images": 1059, "annotations": 2670},
}
MACOS_METADATA = ("__MACOSX/", "/._", ".DS_Store")


def sha256sum(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_dataset_root(extract_dir: Path) -> Path | None:
    roots = [p.parent.parent for p in extract_dir.rglob("_annotations.coco.json")
             if p.parent.name == "train" and "__MACOSX" not in p.parts]
    return min(roots, key=lambda p: len(p.parts)) if roots else None


def is_complete(root: Path | None) -> bool:
    if root is None:
        return False
    for split in SPLITS:
        annotation_file = root / split / "_annotations.coco.json"
        if not annotation_file.exists():
            return False
        with open(annotation_file, encoding="utf-8") as fh:
            expected = {image["file_name"] for image in json.load(fh)["images"]}
        if not expected <= {p.name for p in (root / split).iterdir()}:
            return False
    return True


def prepare_dataset(zip_path: Path, extract_dir: Path, reference_sha256: str) -> Path:
    if reference_sha256:
        actual = sha256sum(zip_path)
        if actual == reference_sha256:
            print("Checksum sama dengan zip referensi tim.")
        else:
            print(f"Checksum berbeda dari zip referensi tim ({actual[:12]}...); isi divalidasi lewat jumlah data.")

    root = find_dataset_root(extract_dir) if extract_dir.exists() else None
    if is_complete(root):
        print(f"Dataset sudah terekstrak di {root}")
        return root

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        members = [m for m in zf.infolist() if not any(tag in "/" + m.filename for tag in MACOS_METADATA)]
        zf.extractall(extract_dir, members=members)

    root = find_dataset_root(extract_dir)
    if not is_complete(root):
        raise RuntimeError(f"Folder train/, valid/, dan test/ yang lengkap tidak ditemukan di {DATA_ZIP}")
    print(f"Dataset diekstrak ke {root}")
    return root


def load_coco(split: str) -> dict:
    with open(RAW_DIR / split / "_annotations.coco.json", encoding="utf-8") as fh:
        return json.load(fh)


RAW_DIR = prepare_dataset(Path(DATA_ZIP), EXTRACT_DIR, DATA_ZIP_SHA256)
coco = {split: load_coco(split) for split in SPLITS}

for split, expected in EXPECTED_COUNTS.items():
    n_images, n_annotations = len(coco[split]["images"]), len(coco[split]["annotations"])
    assert n_images == expected["images"], f"{split}: {n_images} gambar"
    assert n_annotations == expected["annotations"], f"{split}: {n_annotations} anotasi"
    print(f"{split:<6} {n_images:>6} gambar  {n_annotations:>6} anotasi")

## 3. Pemetaan 21 kategori → 11 kelas canonical

File COCO bawaan memiliki 21 kategori: 18 nama kelas dengan variasi penulisan dan 3 supercategory (`Leaf-blight`, `Rice-Leaf-Diseasee`, `paddy`) yang tidak memiliki bounding box. Pemetaan dilakukan berdasarkan nama kategori, lalu divalidasi agar tidak ada anotasi yang tertinggal.

Urutan kelas mengikuti urutan alfabet pada halaman regulasi dan dipakai konsisten di training, evaluasi, dan inference.

In [ ]:
CLASS_NAMES = [
    "Bacterial leaf blight",
    "Bacterial panicle blight",
    "Blast",
    "Brown spot",
    "False smut",
    "Healthy",
    "Leaf roller",
    "Leaf scald",
    "Narrow brown",
    "Sheath blight",
    "Tungro",
]

RAW_TO_CANONICAL = {
    "Bacterial leaf blight": "Bacterial leaf blight",
    "Bacterial panicle Blight": "Bacterial panicle blight",
    "Blast": "Blast",
    "Leaf blast": "Blast",
    "Infected Blast": "Blast",
    "BrownSpot": "Brown spot",
    "Brown spot": "Brown spot",
    "False-Smut": "False smut",
    "Healthy Rice Leaf": "Healthy",
    "Healthy Rice beads": "Healthy",
    "Healthy": "Healthy",
    "healthy": "Healthy",
    "Leaf-roller": "Leaf roller",
    "Leaf Scald": "Leaf scald",
    "Leaf scald": "Leaf scald",
    "Narrow brown": "Narrow brown",
    "Sheath Blight": "Sheath blight",
    "Rice-Tungro": "Tungro",
}
SUPERCATEGORIES = {"Leaf-blight", "Rice-Leaf-Diseasee", "paddy"}
CLASS_INDEX = {name: idx for idx, name in enumerate(CLASS_NAMES)}


def to_frames(data: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Returns (images, boxes); boxes use canonical class ids and COCO xywh in pixels.
    category_names = {c["id"]: c["name"] for c in data["categories"]}
    unknown = set(category_names.values()) - set(RAW_TO_CANONICAL) - SUPERCATEGORIES
    if unknown:
        raise ValueError(f"Kategori tanpa pemetaan: {sorted(unknown)}")

    images = pd.DataFrame(data["images"])[["id", "file_name", "width", "height"]]
    rows = []
    for ann in data["annotations"]:
        raw_name = category_names[ann["category_id"]]
        if raw_name in SUPERCATEGORIES:
            continue
        x, y, w, h = ann["bbox"]
        rows.append((ann["image_id"], CLASS_INDEX[RAW_TO_CANONICAL[raw_name]], x, y, w, h))
    boxes = pd.DataFrame(rows, columns=["image_id", "class_id", "x", "y", "w", "h"])
    boxes = boxes.merge(images, left_on="image_id", right_on="id").drop(columns="id")
    return images, boxes


frames = {split: to_frames(coco[split]) for split in SPLITS}

for split in SPLITS:
    _, boxes = frames[split]
    assert len(boxes) == EXPECTED_COUNTS[split]["annotations"], split

class_counts = pd.DataFrame(
    {split: frames[split][1]["class_id"].value_counts().sort_index() for split in SPLITS}
).set_axis(CLASS_NAMES)
class_counts.loc["Total"] = class_counts.sum()
class_counts

## 4. Pembersihan anotasi

Pembersihan hanya diterapkan pada data **train**. Label valid dan test dipakai apa adanya (hanya dipetakan ke 11 kelas) supaya skor yang kita hitung sama dengan perhitungan juri.

| Aturan | Tindakan |
|---|---|
| Koordinat di luar batas gambar | Dipotong ke batas gambar |
| Sisi box < 2 px | Dibuang (luas 1–3 px², hampir pasti label rusak) |
| Box identik dalam satu gambar | Disisakan satu |
| Box seluas gambar penuh | Dipertahankan (gaya label sebagian sumber data) |
| Gambar tanpa anotasi | Dipertahankan sebagai gambar latar |

In [ ]:
MIN_BOX_SIDE = 2.0


def clip_boxes(boxes: pd.DataFrame) -> pd.DataFrame:
    out = boxes.copy()
    x1 = out["x"].clip(lower=0)
    y1 = out["y"].clip(lower=0)
    x2 = (out["x"] + out["w"]).clip(upper=out["width"])
    y2 = (out["y"] + out["h"]).clip(upper=out["height"])
    out["x"], out["y"], out["w"], out["h"] = x1, y1, x2 - x1, y2 - y1
    return out


def clean_train_boxes(boxes: pd.DataFrame) -> pd.DataFrame:
    clipped = clip_boxes(boxes)
    too_small = (clipped["w"] < MIN_BOX_SIDE) | (clipped["h"] < MIN_BOX_SIDE)
    kept = clipped[~too_small]
    deduped = kept.drop_duplicates(subset=["image_id", "class_id", "x", "y", "w", "h"])
    print(f"train: {len(boxes)} box → {len(deduped)} box "
          f"(sisi < {MIN_BOX_SIDE:g}px: {int(too_small.sum())}, duplikat: {len(kept) - len(deduped)})")
    return deduped


if MODE == "full":
    train_images, train_boxes = frames["train"]
    train_boxes = clean_train_boxes(train_boxes)
    valid_images, valid_boxes = frames["valid"][0], clip_boxes(frames["valid"][1])

## 5. Konversi ke format YOLO

Setiap gambar ditautkan (hard link, atau disalin bila tidak didukung) ke struktur `images/<split>` dan labelnya ditulis ke `labels/<split>` dalam format `class cx cy w h` ternormalisasi.

Untuk run dengan penyeimbangan kelas, gambar train diulang menurut *repeat factor sampling*: kelas yang muncul di sedikit gambar mendapat faktor `r_c = max(1, sqrt(t / f_c))`, dan setiap gambar diulang sebanyak faktor terbesar dari kelas di dalamnya. Pembulatan memakai RNG dengan seed tetap. Hanya data train yang dipakai untuk menghitung faktor ini.

In [ ]:
def link_or_copy(src: Path, dst: Path) -> None:
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


def yolo_lines(image_boxes: pd.DataFrame) -> list[str]:
    lines = []
    for row in image_boxes.itertuples(index=False):
        cx = (row.x + row.w / 2) / row.width
        cy = (row.y + row.h / 2) / row.height
        lines.append(f"{row.class_id} {cx:.6f} {cy:.6f} {row.w / row.width:.6f} {row.h / row.height:.6f}")
    return lines


def export_split(split: str, images: pd.DataFrame, boxes: pd.DataFrame, out_root: Path,
                 repeats: dict[str, int] | None = None) -> int:
    image_dir, label_dir = out_root / "images" / split, out_root / "labels" / split
    for folder in (image_dir, label_dir):
        shutil.rmtree(folder, ignore_errors=True)
        folder.mkdir(parents=True)

    boxes_by_file = dict(tuple(boxes.groupby("file_name")))
    empty = boxes.iloc[0:0]
    written = 0
    for file_name in sorted(images["file_name"]):
        lines = "\n".join(yolo_lines(boxes_by_file.get(file_name, empty)))
        stem, suffix = Path(file_name).stem, Path(file_name).suffix
        for k in range((repeats or {}).get(file_name, 1)):
            name = stem if k == 0 else f"{stem}__rep{k}"
            link_or_copy(RAW_DIR / split / file_name, image_dir / f"{name}{suffix}")
            (label_dir / f"{name}.txt").write_text(lines)
            written += 1
    return written


def repeat_factors(images: pd.DataFrame, boxes: pd.DataFrame, threshold: float, seed: int) -> dict[str, int]:
    classes_per_image = boxes.groupby("file_name")["class_id"].agg(set)
    image_freq = {c: sum(c in s for s in classes_per_image) / len(images) for c in range(len(CLASS_NAMES))}
    class_factor = {c: max(1.0, math.sqrt(threshold / f)) if f > 0 else 1.0 for c, f in image_freq.items()}

    rng = np.random.default_rng(seed)
    repeats = {}
    for file_name in sorted(images["file_name"]):
        classes = classes_per_image.get(file_name, set())
        factor = max((class_factor[c] for c in classes), default=1.0)
        repeats[file_name] = int(factor) + int(rng.random() < factor - int(factor))
    return repeats

## 6. Konfigurasi dataset

`data.yaml` hanya berisi train dan valid. Test ditulis ke file terpisah (`data_test.yaml`) di bagian 10–11, sehingga test tidak pernah terdaftar di konfigurasi training.

In [ ]:
def write_data_yaml(path: Path, root: Path, val_split: str = "valid") -> Path:
    config = {
        "path": str(root),
        "train": "images/train",
        "val": f"images/{val_split}",
        "names": dict(enumerate(CLASS_NAMES)),
    }
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path

## 7. Training

Setiap run hanya berbeda satu hal dari baseline:

| Run | Model | Perbedaan |
|---|---|---|
| `E0_smoke` | YOLO11s | Uji alur: 2 epoch, 5% data |
| `A_yolo11s_s42` | YOLO11s | Baseline |
| `B_yolo26s_s42` | YOLO26s | Arsitektur |
| `C_yolo11s_bal_s42` | YOLO11s | Penyeimbangan kelas (repeat factor sampling) |
| `A800_yolo11s_s42` | YOLO11s | Lanjutan dari `best.pt` Run A pada resolusi 800 untuk lesi kecil |

`A800_yolo11s_s42` tidak dimulai dari nol: bobot awalnya adalah hasil Run A milik tim sendiri, yang dilatih dari nol pada dataset resmi. Karena itu run ini memakai `pretrained=True`, yang di Ultralytics berarti *pakai bobot dari file `.pt` yang dimuat*; tanpa itu bobot Run A akan dibuang. Tidak ada bobot eksternal yang terlibat. Optimizer ditulis eksplisit (SGD, `lr0=0.002`) karena mode `auto` mengabaikan `lr0`, sedangkan fine-tuning butuh learning rate yang lebih kecil dari training awal. Run ini membutuhkan folder Run A (`RUNS_DIR/A_yolo11s_s42`) di Drive yang sama, misalnya lewat shortcut.

Output run ditulis ke Google Drive (`RUNS_DIR/RUN_ID`), termasuk checkpoint tiap 10 epoch. Kalau sesi Colab terputus, jalankan ulang notebook dengan `RUN_ID` yang sama: training dilanjutkan dari `last.pt`.

In [ ]:
TRAIN_ARGS = dict(
    imgsz=IMGSZ,
    epochs=150,
    batch=16,
    seed=SEED,
    deterministic=True,
    pretrained=False,
    workers=2,
    cache=False,
    save_period=10,
    patience=0,
    close_mosaic=10,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    flipud=0.5,
    plots=True,
)

RUNS = {
    "E0_smoke": {"model": "yolo11s.yaml", "balanced": False, "overrides": {"epochs": 2, "fraction": 0.05}},
    "A_yolo11s_s42": {"model": "yolo11s.yaml", "balanced": False},
    "B_yolo26s_s42": {"model": "yolo26s.yaml", "balanced": False},
    "C_yolo11s_bal_s42": {"model": "yolo11s.yaml", "balanced": True, "repeat_threshold": 0.10},
    "A800_yolo11s_s42": {
        "model": RUNS_DIR / "A_yolo11s_s42" / "weights" / "best.pt",
        "balanced": False,
        "overrides": {"imgsz": 800, "epochs": 40, "pretrained": True, "optimizer": "SGD",
                      "lr0": 0.002, "lrf": 0.1, "warmup_epochs": 1.0},
    },
}


def prepare_training_data(run: dict) -> Path:
    repeats = None
    if run["balanced"]:
        repeats = repeat_factors(train_images, train_boxes, run["repeat_threshold"], SEED)
    n_train = export_split("train", train_images, train_boxes, YOLO_DIR, repeats)
    n_valid = export_split("valid", valid_images, valid_boxes, YOLO_DIR)
    print(f"train: {n_train} gambar, valid: {n_valid} gambar")
    return write_data_yaml(WORK_DIR / "data.yaml", YOLO_DIR)


def training_finished(checkpoint: Path) -> bool:
    state = torch.load(checkpoint, map_location="cpu", weights_only=False)
    return state.get("epoch", -1) == -1


def train(run_id: str) -> Path:
    run = RUNS[run_id]
    run_dir = RUNS_DIR / run_id
    last = run_dir / "weights" / "last.pt"
    data_yaml = prepare_training_data(run)

    if last.exists() and training_finished(last):
        print(f"{run_id} sudah selesai.")
    elif last.exists():
        print(f"Melanjutkan {run_id} dari {last}")
        YOLO(last).train(resume=True)
    else:
        if isinstance(run["model"], Path) and not run["model"].exists():
            raise FileNotFoundError(f"Bobot awal tidak ditemukan: {run['model']}")
        model = YOLO(str(run["model"]))
        model.train(data=str(data_yaml), project=str(RUNS_DIR), name=run_id, exist_ok=True,
                    **{**TRAIN_ARGS, **run.get("overrides", {})})
    return run_dir


if MODE == "full":
    run_dir = train(RUN_ID)

## 8. Evaluasi checkpoint di valid

Skor dihitung dalam dua versi:

- **E1** — rata-rata F1 per kelas dari `model.val()`. Ultralytics mengambil F1 pada satu confidence yang memaksimalkan rata-rata F1 semua kelas (IoU 0,5).
- **E2** — F1 makro 11 kelas pada confidence tetap `E2_CONF`, dihitung sendiri: prediksi diurutkan dari confidence tertinggi lalu dicocokkan secara greedy ke label dengan IoU ≥ 0,5 per kelas. Nilai `E2_CONF` dipilih dari valid.

mAP@50 ikut dicatat sebagai pembanding. Semua parameter evaluasi ditulis eksplisit (FP32, batch tetap). Setiap checkpoint dievaluasi pada resolusi saat ia dilatih (640, atau 800 untuk `A800_yolo11s_s42`), yang dibaca dari metadata checkpoint.

In [ ]:
VAL_ARGS = dict(batch=EVAL_BATCH, conf=0.001, iou=0.7, max_det=300, quantize=32,
                plots=False, verbose=False, workers=2)


def trained_imgsz(model: YOLO) -> int:
    # Evaluate each checkpoint at the resolution it was trained on (640 or 800).
    return int(model.overrides.get("imgsz", IMGSZ))


def per_class_f1(metrics) -> np.ndarray:
    scores = np.full(len(CLASS_NAMES), np.nan)
    scores[metrics.box.ap_class_index] = metrics.box.f1
    return scores


def evaluate_e1(weights: Path, data_yaml: Path) -> dict:
    model = YOLO(weights)
    metrics = model.val(data=str(data_yaml), split="val", imgsz=trained_imgsz(model), **VAL_ARGS)
    f1 = per_class_f1(metrics)
    return {"E1": float(np.nanmean(f1)), "mAP50": float(metrics.box.map50),
            **{f"F1 {name}": score for name, score in zip(CLASS_NAMES, f1)}}


def evaluate_run(run_id: str, data_yaml: Path) -> pd.DataFrame:
    weights_dir = RUNS_DIR / run_id / "weights"
    checkpoints = sorted(weights_dir.glob("epoch*.pt"), key=lambda p: int(p.stem[5:]))
    checkpoints += [weights_dir / name for name in ("last.pt", "best.pt") if (weights_dir / name).exists()]
    rows = [{"run_id": run_id, "checkpoint": ckpt.name, **evaluate_e1(ckpt, data_yaml)} for ckpt in checkpoints]
    table = pd.DataFrame(rows).sort_values("E1", ascending=False)
    table.to_csv(RUNS_DIR / run_id / "checkpoint_eval_valid.csv", index=False)
    return table


if MODE == "full":
    valid_table = evaluate_run(RUN_ID, WORK_DIR / "data.yaml")
    display(valid_table.head(10))

In [ ]:
def ground_truth(images: pd.DataFrame, boxes: pd.DataFrame) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    gt = {}
    grouped = dict(tuple(boxes.groupby("file_name")))
    for file_name in images["file_name"]:
        b = grouped.get(file_name)
        if b is None:
            gt[file_name] = (np.zeros(0, dtype=int), np.zeros((0, 4)))
        else:
            xyxy = np.stack([b["x"], b["y"], b["x"] + b["w"], b["y"] + b["h"]], axis=1)
            gt[file_name] = (b["class_id"].to_numpy(), xyxy)
    return gt


def predict_split(weights: Path, image_dir: Path, file_names: list[str]) -> dict[str, tuple]:
    model = YOLO(weights)
    preds = {}
    paths = [str(image_dir / name) for name in file_names]
    for start in range(0, len(paths), EVAL_BATCH):
        batch = paths[start:start + EVAL_BATCH]
        results = model.predict(batch, imgsz=trained_imgsz(model), conf=0.001, iou=0.7, max_det=300, quantize=32, verbose=False)
        for path, res in zip(batch, results):
            boxes = res.boxes
            preds[Path(path).name] = (boxes.cls.cpu().numpy().astype(int), boxes.conf.cpu().numpy(),
                                      boxes.xyxy.cpu().numpy())
    return preds


def box_iou(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    tl = np.maximum(a[:, None, :2], b[None, :, :2])
    br = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.prod(np.clip(br - tl, 0, None), axis=2)
    area_a = np.prod(a[:, 2:] - a[:, :2], axis=1)
    area_b = np.prod(b[:, 2:] - b[:, :2], axis=1)
    return inter / (area_a[:, None] + area_b[None, :] - inter + 1e-9)


def match_predictions(preds: dict, gt: dict, iou_threshold: float = 0.5):
    # Greedy matching per class in descending confidence. Truncating the sorted list at any
    # confidence threshold keeps the same matches, so one pass serves every threshold.
    classes, confs, hits = [], [], []
    n_gt = np.zeros(len(CLASS_NAMES), dtype=int)
    for file_name, (gt_cls, gt_xyxy) in gt.items():
        n_gt += np.bincount(gt_cls, minlength=len(CLASS_NAMES))
        p_cls, p_conf, p_xyxy = preds[file_name]
        for c in np.unique(p_cls):
            p_idx = np.where(p_cls == c)[0]
            p_idx = p_idx[np.argsort(-p_conf[p_idx], kind="stable")]
            g_idx = np.where(gt_cls == c)[0]
            matched = np.zeros(len(g_idx), dtype=bool)
            ious = box_iou(p_xyxy[p_idx], gt_xyxy[g_idx]) if len(g_idx) else np.zeros((len(p_idx), 0))
            for row, i in enumerate(p_idx):
                hit = False
                if len(g_idx):
                    candidates = np.where(~matched & (ious[row] >= iou_threshold))[0]
                    if len(candidates):
                        matched[candidates[np.argmax(ious[row, candidates])]] = True
                        hit = True
                classes.append(c)
                confs.append(p_conf[i])
                hits.append(hit)
    return np.array(classes, dtype=int), np.array(confs), np.array(hits, dtype=bool), n_gt


def macro_f1_at(matches, conf_threshold: float) -> tuple[float, np.ndarray]:
    classes, confs, hits, n_gt = matches
    keep = confs >= conf_threshold
    tp = np.bincount(classes[keep & hits], minlength=len(CLASS_NAMES))
    fp = np.bincount(classes[keep & ~hits], minlength=len(CLASS_NAMES))
    fn = n_gt - tp
    f1 = np.where(n_gt > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan)
    return float(np.nanmean(f1)), f1


def best_conf(matches, grid=np.round(np.arange(0.05, 0.951, 0.01), 2)) -> tuple[float, float]:
    scores = [macro_f1_at(matches, t)[0] for t in grid]
    i = int(np.argmax(scores))
    return float(grid[i]), float(scores[i])

## 9. Pemilihan model final

Kandidat diurutkan berdasarkan E1 di valid (tabel dari bagian 8, digabung dari semua run). Dari kandidat teratas dipilih yang kurva F1-nya landai di sekitar puncak dan tidak memiliki kelas dengan F1 sangat rendah. Split test tidak dilibatkan dalam keputusan ini.

Isi `SELECTED_WEIGHTS` dengan checkpoint terpilih. Sel ini menghitung `E2_CONF` dari valid; salin nilainya ke sel konfigurasi.

In [ ]:
SELECTED_WEIGHTS = None  # contoh: RUNS_DIR / "A_yolo11s_s42" / "weights" / "epoch140.pt"

if MODE == "full" and SELECTED_WEIGHTS is not None:
    valid_gt = ground_truth(valid_images, valid_boxes)
    valid_preds = predict_split(Path(SELECTED_WEIGHTS), RAW_DIR / "valid", list(valid_gt))
    valid_matches = match_predictions(valid_preds, valid_gt)
    conf, score = best_conf(valid_matches)
    curve = pd.Series({t: macro_f1_at(valid_matches, t)[0] for t in (conf - 0.1, conf - 0.05, conf, conf + 0.05, conf + 0.1)})
    print(f"E2_CONF = {conf:.2f}  (F1 makro valid {score:.4f})")
    print("Kelandaian kurva F1 di sekitar puncak:")
    print(curve.round(4).to_string())